# RL DRC Corrector — Evaluate & Use

Loads the trained model and tests it on all 15 circuits.
Shows DRC errors before and after RL correction.

**Run `00_train.ipynb` first** to generate `drc_corrector_ppo.zip`.

In [2]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))

import numpy as np
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel
from stable_baselines3 import PPO

print('All imports OK')
print(f'Model: {os.path.abspath("../../src/gelochip/gl/drc_corrector_ppo.pt")}')

All imports OK
Model: /home/irman/Gelochip/src/gelochip/gl/drc_corrector_ppo.pt


## Builder functions

Same 13 builder functions used in `00_train.ipynb`.

In [3]:
# ── helpers ─────────────────────────────────────────────────────────────────
def lp(p):
    return dict(with_tie=p.get('with_tie', False), with_dummy=p.get('with_dummy', False))

def bp(p):
    return dict(placement=p.get('placement', 'column'), sep_mult=p.get('sep_mult', 2.0),
                met_layer=p.get('met_layer', 1), width_mult=p.get('width_mult', 1.0))

# ── 01 Inverter ──────────────────────────────────────────────────────────────
def build_inverter(cp, p, name='inverter'):
    vin = gl.Net('vin'); vout = gl.Net('vout')
    mn = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vin, d=vout, s=gl.gnd, **lp(p))
    mp = gl.pmos(w=cp['wp'], fingers=cp['fn'], g=vin, d=vout, s=gl.vdd, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))

# ── 02 Current Mirror ─────────────────────────────────────────────────────────
def build_current_mirror(cp, p, name='cmirror'):
    vbias = gl.Net('vbias'); iout = gl.Net('iout')
    fet  = gl.nmos if cp.get('n_or_p', 'n') == 'n' else gl.pmos
    rail = gl.gnd  if cp.get('n_or_p', 'n') == 'n' else gl.vdd
    m_ref  = fet(w=cp['w'], fingers=1,                  g=vbias, d=vbias, s=rail, **lp(p))
    m_copy = fet(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=iout,  s=rail, **lp(p))
    return gl.build(m_ref, m_copy, name=name, **bp(p))

# ── 03 Differential Pair ─────────────────────────────────────────────────────
def build_diff_pair(cp, p, name='diff_pair'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vop = gl.Net('vop'); vom = gl.Net('vom'); vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2,       g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vim,   d=vom,   s=vtail,  **lp(p))
    return gl.build(mt, mn_p, mn_m, name=name, **bp(p))

# ── 04 OTA (5T) ──────────────────────────────────────────────────────────────
def build_ota(cp, p, name='ota'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vout = gl.Net('vout'); vleft = gl.Net('vleft'); vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2,       g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'], fingers=cp['fn'], g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'], fingers=cp['fn'], g=vleft, d=vout,  s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))

# ── 05 Flipped Voltage Follower ───────────────────────────────────────────────
def build_fvf(cp, p, name='fvf'):
    vin = gl.Net('vin'); vout = gl.Net('vout'); vfb = gl.Net('vfb')
    mn = gl.nmos(w=cp['w_main'], fingers=2, g=vfb, d=vout, s=gl.gnd, **lp(p))
    mp = gl.pmos(w=cp['w_fb'],  fingers=1, g=vin, d=vfb,  s=gl.vdd, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))

# ── 06 Transmission Gate ──────────────────────────────────────────────────────
def build_tgate(cp, p, name='tgate'):
    vin = gl.Net('vin'); vout = gl.Net('vout')
    vctrl = gl.Net('vctrl'); vctrl_n = gl.Net('vctrl_n')
    mn = gl.nmos(w=cp['wn'], fingers=1, g=vctrl,   d=vout, s=vin, **lp(p))
    mp = gl.pmos(w=cp['wp'], fingers=1, g=vctrl_n, d=vout, s=vin, **lp(p))
    return gl.build(mn, mp, name=name, **bp(p))

# ── 07 Stacked Current Mirror ─────────────────────────────────────────────────
def build_stacked_cmirror(cp, p, name='stacked_cm'):
    vbias = gl.Net('vbias'); iout = gl.Net('iout'); vcasc = gl.Net('vcasc')
    m_ref = gl.nmos(w=cp['w'], fingers=1,                  g=vbias, d=vcasc, s=gl.gnd, **lp(p))
    m_csc = gl.nmos(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=vbias, s=vcasc,  **lp(p))
    m_out = gl.nmos(w=cp['w'], fingers=cp.get('ratio', 1), g=vbias, d=iout,  s=gl.gnd, **lp(p))
    return gl.build(m_ref, m_csc, m_out, name=name, **bp(p))

# ── 08 Low-Voltage Current Mirror ─────────────────────────────────────────────
def build_lvcmirror(cp, p, name='lvcm'):
    vbias = gl.Net('vbias'); iout = gl.Net('iout'); vx = gl.Net('vx')
    m_ref  = gl.nmos(w=cp['w'],        fingers=1, g=vbias, d=vbias, s=gl.gnd, **lp(p))
    m_aux  = gl.nmos(w=cp['w_narrow'], fingers=1, g=vbias, d=vx,    s=gl.gnd, **lp(p))
    m_copy = gl.nmos(w=cp['w'],        fingers=1, g=vbias, d=iout,  s=gl.gnd, **lp(p))
    return gl.build(m_ref, m_aux, m_copy, name=name, **bp(p))

# ── 09 Diff Pair + CM Bias ────────────────────────────────────────────────────
def build_diff_pair_cmbias(cp, p, name='dp_cmbias'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vop = gl.Net('vop'); vom = gl.Net('vom'); vbias = gl.Net('vbias'); vload = gl.Net('vload')
    mt   = gl.nmos(w=cp['wt'], fingers=2,       g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vim,   d=vom,   s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wn'], fingers=cp['fn'], g=vload, d=vload, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wn'], fingers=cp['fn'], g=vload, d=vop,   s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))

# ── 10 Diff Pair + Stacked Bias ───────────────────────────────────────────────
def build_diff_pair_stacked(cp, p, name='dp_stacked'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vop = gl.Net('vop'); vom = gl.Net('vom'); vbias = gl.Net('vbias'); vcasc = gl.Net('vcasc')
    mt   = gl.nmos(w=cp['w'],  fingers=2,       g=vbias, d=vcasc, s=gl.gnd, **lp(p))
    mc   = gl.nmos(w=cp['w'],  fingers=2,       g=vbias, d=vtail, s=vcasc,  **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vip,   d=vop,   s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=cp['fn'], g=vim,   d=vom,   s=vtail,  **lp(p))
    return gl.build(mt, mc, mn_p, mn_m, name=name, **bp(p))

# ── 11 Two-Stage OTA ─────────────────────────────────────────────────────────
def build_twostage_ota(cp, p, name='twostage_ota'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vout = gl.Net('vout'); vleft = gl.Net('vleft'); vbias = gl.Net('vbias'); vcs = gl.Net('vcs')
    mt   = gl.nmos(w=cp['wt'],  fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'],  fingers=2, g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'],  fingers=2, g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'],  fingers=2, g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'],  fingers=2, g=vleft, d=vout,  s=gl.vdd, **lp(p))
    mcs  = gl.nmos(w=cp['wcs'], fingers=2, g=vout,  d=vcs,   s=gl.gnd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, mcs, name=name, **bp(p))

# ── 12 P-Block ────────────────────────────────────────────────────────────────
def build_p_block(cp, p, name='p_block'):
    vbias = gl.Net('vbias'); iout = gl.Net('iout'); vtail = gl.Net('vtail')
    mp_ref = gl.pmos(w=cp['wp'], fingers=2, g=vbias, d=vbias, s=gl.vdd, **lp(p))
    mp_out = gl.pmos(w=cp['wp'], fingers=2, g=vbias, d=iout,  s=gl.vdd, **lp(p))
    mt     = gl.nmos(w=cp['wt'], fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_out = gl.nmos(w=cp['wn'], fingers=2, g=vbias, d=iout,  s=vtail,  **lp(p))
    return gl.build(mp_ref, mp_out, mt, mn_out, name=name, **bp(p))

# ── 13 Differential-to-Single Ended ──────────────────────────────────────────
def build_diff_to_single(cp, p, name='d2s'):
    vip = gl.Net('vip'); vim = gl.Net('vim'); vtail = gl.Net('vtail')
    vout = gl.Net('vout'); vleft = gl.Net('vleft'); vbias = gl.Net('vbias')
    mt   = gl.nmos(w=cp['wt'], fingers=2, g=vbias, d=vtail, s=gl.gnd, **lp(p))
    mn_p = gl.nmos(w=cp['wn'], fingers=2, g=vip,   d=vout,  s=vtail,  **lp(p))
    mn_m = gl.nmos(w=cp['wn'], fingers=2, g=vim,   d=vleft, s=vtail,  **lp(p))
    mp_l = gl.pmos(w=cp['wp'], fingers=2, g=vleft, d=vleft, s=gl.vdd, **lp(p))
    mp_r = gl.pmos(w=cp['wp'], fingers=2, g=vleft, d=vout,  s=gl.vdd, **lp(p))
    return gl.build(mt, mn_p, mn_m, mp_l, mp_r, name=name, **bp(p))

import random
CIRCUIT_POOL = [
    (build_inverter,          {'wn': 2.0, 'wp': 4.0, 'fn': 2},                 'inverter'),
    (build_inverter,          {'wn': 4.0, 'wp': 8.0, 'fn': 4},                 'inverter_large'),
    (build_current_mirror,    {'w': 4.0, 'ratio': 1, 'n_or_p': 'n'},           'cmirror_n'),
    (build_current_mirror,    {'w': 4.0, 'ratio': 2, 'n_or_p': 'p'},           'cmirror_p'),
    (build_diff_pair,         {'wn': 3.0, 'wt': 4.0, 'fn': 2},                 'diff_pair'),
    (build_fvf,               {'w_main': 6.6, 'w_fb': 3.3},                     'fvf'),
    (build_tgate,             {'wn': 2.0, 'wp': 2.0},                           'tgate'),
    (build_ota,               {'wn': 3.0, 'wp': 4.0, 'wt': 4.0, 'fn': 2},      'ota'),
    (build_stacked_cmirror,   {'w': 4.0, 'ratio': 1},                           'stacked_cm'),
    (build_lvcmirror,         {'w': 4.0, 'w_narrow': 1.5},                      'lvcm'),
    (build_diff_pair_cmbias,  {'wn': 3.0, 'wt': 4.0, 'fn': 2},                 'dp_cmbias'),
    (build_diff_pair_stacked, {'wn': 3.0, 'w': 4.0, 'fn': 2},                  'dp_stacked'),
    (build_twostage_ota,      {'wn': 3.0, 'wp': 4.0, 'wt': 4.0, 'wcs': 8.0},  'twostage_ota'),
    (build_p_block,           {'wp': 3.0, 'wn': 4.0, 'wt': 4.0},               'p_block'),
    (build_diff_to_single,    {'wn': 3.0, 'wp': 4.0, 'wt': 4.0},               'd2s'),
]

print(f'Builders ready. Circuit pool: {len(CIRCUIT_POOL)} circuits')

Builders ready. Circuit pool: 15 circuits


## Test on all 15 circuits

For each circuit:
1. Build with **default params** → count naive DRC errors
2. Run **RL corrector** → find best layout params
3. Guaranteed fallback: `with_tie=True` sweep (always fixes N-well DRC)

In [4]:
MODEL_PATH = os.path.join('..', '..', 'src', 'gelochip', 'gl', 'drc_corrector_ppo')
import torch
_ckpt = torch.load(MODEL_PATH + '.pt', weights_only=False)

results = {}

for build_fn, circuit_params, label in CIRCUIT_POOL:
    print(f'\n── {label} ──────────────────────────────')

    # 1. Naive: default layout params
    try:
        naive_lp = dict(gl.DEFAULT_LAYOUT_PARAMS)
        naive = build_fn(circuit_params, naive_lp, f'{label}_naive')
        naive_drc = naive.drc(silent=True)
        naive_n = naive_drc.get('total_errors', 0)
    except Exception as e:
        print(f'  naive build error: {e}')
        naive_n = -1

    # 2. RL corrector
    env   = gl.make_env(build_fn, circuit_params, max_steps=15)
    from stable_baselines3 import PPO as _PPO
    model = _PPO('MlpPolicy', env, policy_kwargs=dict(net_arch=[128, 128, 64]))
    model.policy.load_state_dict(_ckpt['policy_state_dict'])
    model.policy.set_training_mode(False)
    obs, info = env.reset()
    for _ in range(15):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
        if done:
            break

    best_lp = env.best_layout_params
    fixed_n = env._best_errors

    # 3. Fallback: best params embedded in .pt checkpoint
    if fixed_n > 0 and 'best_layout_params' in _ckpt:
        try:
            slp = _ckpt['best_layout_params']
            sd = build_fn(circuit_params, slp, '_rl_tmp').drc(silent=True)
            sn = sd.get('total_errors', 999)
            if sn < fixed_n:
                best_lp = slp
                fixed_n = sn
        except Exception:
            pass

    # 4. Guaranteed fallback: with_tie=True sweep (fixes NW violations)
    if fixed_n > 0:
        for _pl in ('column', 'row'):
            for _sep in (2.0, 2.5, 1.5, 3.0):
                _lp = dict(gl.DEFAULT_LAYOUT_PARAMS, with_tie=True, with_dummy=True,
                           placement=_pl, sep_mult=_sep)
                try:
                    _d = build_fn(circuit_params, _lp, '_rl_tmp').drc(silent=True)
                    _n = _d.get('total_errors', 999)
                    if _n < fixed_n:
                        fixed_n = _n
                        best_lp = _lp
                    if fixed_n == 0:
                        break
                except Exception:
                    pass
            if fixed_n == 0:
                break

    results[label] = {'naive': naive_n, 'fixed': fixed_n, 'lp': best_lp}
    icon = '✅' if fixed_n == 0 else '❌'
    print(f'  {icon} naive={naive_n:3d} → fixed={fixed_n:3d}  '
          f'tie={int(best_lp["with_tie"])} dum={int(best_lp["with_dummy"])} '
          f'{best_lp["placement"]} sep={best_lp["sep_mult"]:.1f} '
          f'met={best_lp.get("met_layer",1)} w={best_lp.get("width_mult",1.0):.1f}')

2026-05-20 14:53:50.648 | INFO     | gdsfactory.pdk:activate:337 - 'gf180' PDK is now active



── inverter ──────────────────────────────


2026-05-20 14:53:51.268 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/inverter_naive.gds'
2026-05-20 14:53:51.291 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/notebooks/layoutRL/INVERTER_NAIVE.gds'


using provided pdk_root
Defaulting to stale magic_commands.tcl


2026-05-20 14:53:51.523 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmpadtfipx2/INVERTER_NAIVE.gds'


...

using default pdk_root


2026-05-20 14:59:13.343 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/_rl_tmp.gds'
2026-05-20 14:59:13.344 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/notebooks/layoutRL/_RL_TMP.gds'


using provided pdk_root
Defaulting to stale magic_commands.tcl


2026-05-20 14:59:13.567 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmp05m5jp84/_RL_TMP.gds'



Magic 8.3 revision 644 - Compiled on Sun May 17 10:32:25 PM WIB 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmp79g_gibv/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0
Library name: library
Reading "_RL_TMP".
[INFO]: Loading _RL_T

## Summary table

In [5]:
print()
print(f'{"Circuit":<30} {"Naive DRC":>10} {"RL-fixed":>10} {"Status":>8}')
print('-' * 62)
clean = 0
for label, r in results.items():
    status = 'CLEAN' if r['fixed'] == 0 else f"{r['fixed']} err"
    if r['fixed'] == 0: clean += 1
    icon = '✅' if r['fixed'] == 0 else '❌'
    print(f'{icon} {label:<28} {r["naive"]:>10} {r["fixed"]:>10} {status:>8}')
print('-' * 62)
print(f'DRC-clean: {clean}/{len(results)} circuits')


Circuit                         Naive DRC   RL-fixed   Status
--------------------------------------------------------------
✅ inverter                              6          0    CLEAN
✅ inverter_large                        6          0    CLEAN
✅ cmirror_n                             0          0    CLEAN
✅ cmirror_p                            12          0    CLEAN
✅ diff_pair                             0          0    CLEAN
✅ fvf                                   6          0    CLEAN
✅ tgate                                 6          0    CLEAN
✅ ota                                  12          0    CLEAN
✅ stacked_cm                            0          0    CLEAN
✅ lvcm                                  0          0    CLEAN
✅ dp_cmbias                            12          0    CLEAN
✅ dp_stacked                            0          0    CLEAN
✅ twostage_ota                         12          0    CLEAN
✅ p_block                              12          0    CLEAN
✅ d2s 

## Use in your own notebooks

### Option A — Direct
```python
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel

vin, vout = gl.Net('vin'), gl.Net('vout')
mn = gl.nmos(w=2, fingers=2, g=vin, d=vout, s=gl.gnd, with_tie=True)
mp = gl.pmos(w=4, fingers=2, g=vin, d=vout, s=gl.vdd, with_tie=True)
chip = gl.build(mn, mp, name='inverter')
chip.drc()
```

### Option B — Auto-fix with trained RL
```python
def inv_builder(circuit_p, layout_p, name):
    vin, vout = gl.Net('vin'), gl.Net('vout')
    mn = gl.nmos(w=circuit_p['wn'], fingers=circuit_p['fn'],
                 g=vin, d=vout, s=gl.gnd,
                 with_tie=layout_p['with_tie'], with_dummy=layout_p['with_dummy'])
    mp = gl.pmos(w=circuit_p['wp'], fingers=circuit_p['fn'],
                 g=vin, d=vout, s=gl.vdd,
                 with_tie=layout_p['with_tie'], with_dummy=layout_p['with_dummy'])
    return gl.build(mn, mp, name=name,
                    placement=layout_p['placement'], sep_mult=layout_p['sep_mult'],
                    met_layer=layout_p['met_layer'], width_mult=layout_p['width_mult'])

chip = gl.auto_build(inv_builder, {'wn': 2, 'wp': 4, 'fn': 2}, name='inverter')
chip.drc()
```

## Live demo — auto_build on inverter

In [6]:
vin = gl.Net('vin')
vout = gl.Net('vout')
mn = gl.nmos(w=2.0, fingers=2, g=vin, d=vout, s=gl.gnd)
mp = gl.pmos(w=4.0, fingers=2, g=vin, d=vout, s=gl.vdd)

chip = gl.build(mn, mp, name='inverter_rl_fixed')
chip.show()
print()
chip.drc()

## Best layout params found per circuit

In [7]:
import json
print('Best layout params found by RL per circuit:')
print()
for label, r in results.items():
    print(f'{label}:')
    for k, v in r['lp'].items():
        print(f'  {k} = {v}')
    print()

Best layout params found by RL per circuit:

inverter:
  with_tie = True
  with_dummy = False
  placement = row
  sep_mult = 2.1305570658296347
  met_layer = 1
  width_mult = 1.5

inverter_large:
  with_tie = True
  with_dummy = False
  placement = row
  sep_mult = 2.1305570658296347
  met_layer = 1
  width_mult = 1.5

cmirror_n:
  with_tie = False
  with_dummy = False
  placement = column
  sep_mult = 2.091322347521782
  met_layer = 1
  width_mult = 0.9929195963777602

cmirror_p:
  with_tie = True
  with_dummy = False
  placement = row
  sep_mult = 2.1305570658296347
  met_layer = 1
  width_mult = 1.5

diff_pair:
  with_tie = False
  with_dummy = False
  placement = column
  sep_mult = 2.091322347521782
  met_layer = 1
  width_mult = 0.9929195963777602

fvf:
  with_tie = True
  with_dummy = False
  placement = row
  sep_mult = 2.1305570658296347
  met_layer = 1
  width_mult = 1.5

tgate:
  with_tie = True
  with_dummy = False
  placement = row
  sep_mult = 2.1305570658296347
  met_lay